# Лабораторная 3. Задачи 4.8 и 4.13

**4.8.** Найти координаты векторов репера Френе $(T,N,B)$ для
заданных пространственных кривых в указанных точках:
$r(t)=(t,\frac12t^2,\frac13t^3)$ при $t=1$;
$r(t)=(t\sin t,t\cos t,te^t)$ в начале координат;
$r(t)=(a\cos t,a\sin t,bt)$ в произвольной точке;
$r(t)=(a(t-\sin t),a(1-\cos t),4a\cos\frac t2)$ в произвольной точке.

**4.13.** На бинормалях винтовой линии
$(a\cos t,a\sin t,bt)$ отложены отрезки равной длины. Доказать, что
концы этих отрезков лежат на другой винтовой линии.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["animation.embed_limit"] = 100


def finish_animation(anim, fig):
    html = HTML(anim.to_jshtml())
    plt.close(fig)
    return html


def set_axes_equal_3d(ax):
    x_limits = ax.get_xlim3d()
    y_limits = ax.get_ylim3d()
    z_limits = ax.get_zlim3d()
    x_range = abs(x_limits[1] - x_limits[0])
    y_range = abs(y_limits[1] - y_limits[0])
    z_range = abs(z_limits[1] - z_limits[0])
    radius = 0.5 * max([x_range, y_range, z_range])
    x_middle = np.mean(x_limits)
    y_middle = np.mean(y_limits)
    z_middle = np.mean(z_limits)
    ax.set_xlim3d([x_middle - radius, x_middle + radius])
    ax.set_ylim3d([y_middle - radius, y_middle + radius])
    ax.set_zlim3d([z_middle - radius, z_middle + radius])


def setup_3d(ax, xlim, ylim, zlim, title):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(*zlim)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title(title)

In [ ]:
EPS = 1e-9


def normalize(v):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    if n < EPS:
        raise ValueError("Нулевой вектор нельзя нормировать")
    return v / n


def d_curve(r, t, h=1e-5):
    return (np.asarray(r(t + h)) - np.asarray(r(t - h))) / (2 * h)


def dd_curve(r, t, h=1e-4):
    return (np.asarray(r(t + h)) - 2 * np.asarray(r(t)) + np.asarray(r(t - h))) / (h ** 2)


def frenet_frame(r, t):
    p = np.asarray(r(t), dtype=float)
    r1 = d_curve(r, t)
    r2 = dd_curve(r, t)
    T = normalize(r1)
    A_perp = r2 - np.dot(r2, T) * T
    N = normalize(A_perp)
    B = normalize(np.cross(T, N))
    return p, T, N, B


a = 1.5
b = 0.6
curves = {
    "4.8a": (lambda t: np.array([t, 0.5 * t**2, (1 / 3) * t**3]), 1.0, (-1.0, 2.5)),
    "4.8b": (lambda t: np.array([t * np.sin(t), t * np.cos(t), t * np.exp(t)]), 0.0, (-1.4, 1.0)),
    "4.8c": (lambda t: np.array([a * np.cos(t), a * np.sin(t), b * t]), np.pi / 3, (0.0, 4 * np.pi)),
    "4.8d": (lambda t: np.array([a * (t - np.sin(t)), a * (1 - np.cos(t)), 4 * a * np.cos(t / 2)]), np.pi / 2, (0.05, 2 * np.pi - 0.05)),
}

for name, (r, t0, _) in curves.items():
    p, T, N, B = frenet_frame(r, t0)
    print(name)
    print("  точка:", np.round(p, 4))
    print("  T:", np.round(T, 4))
    print("  N:", np.round(N, 4))
    print("  B:", np.round(B, 4))
    print("  скалярные произведения:", round(np.dot(T, N), 8), round(np.dot(T, B), 8), round(np.dot(N, B), 8))

In [ ]:
name = "4.8c"
r, _, t_range = curves[name]
ts = np.linspace(*t_range, 500)
pts = np.array([r(t) for t in ts])

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")


def animate(i):
    ax.clear()
    setup_3d(ax, (-3, 3), (-3, 3), (-1, 9), f"{name}: движущийся репер Френе")
    ax.plot(pts[:, 0], pts[:, 1], pts[:, 2], color="navy", linewidth=2)
    t = t_range[0] + (t_range[1] - t_range[0]) * i / 100
    p, T, N, B = frenet_frame(r, t)
    ax.scatter([p[0]], [p[1]], [p[2]], color="crimson", s=35)
    for vec, label, color in [(T, "T", "crimson"), (N, "N", "darkgreen"), (B, "B", "purple")]:
        q = p + 0.85 * vec
        ax.quiver(p[0], p[1], p[2], 0.85 * vec[0], 0.85 * vec[1], 0.85 * vec[2],
                  color=color, arrow_length_ratio=0.18)
        ax.text(q[0], q[1], q[2], label, color=color)
    ax.view_init(elev=24, azim=40 + i)
    set_axes_equal_3d(ax)
    return []


anim = FuncAnimation(fig, animate, frames=101, interval=70)
finish_animation(anim, fig)

In [ ]:
a = 2.0
b = 0.8
d = 0.9
c = np.sqrt(a**2 + b**2)


def helix(t):
    return np.array([a * np.cos(t), a * np.sin(t), b * t])


def binormal(t):
    return np.vstack([b * np.sin(t) / c, -b * np.cos(t) / c, np.full_like(t, a / c)])


ts = np.linspace(0, 4 * np.pi, 600)
H = helix(ts)
Q = H + d * binormal(ts)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
setup_3d(ax, (-3.2, 3.2), (-3.2, 3.2), (-0.5, 11), "4.13: концы отрезков на бинормалях")
ax.plot(H[0], H[1], H[2], color="navy", linewidth=2, label="исходная винтовая линия")
ax.plot(Q[0], Q[1], Q[2], color="darkorange", linewidth=2, label="новая винтовая линия")
segment, = ax.plot([], [], [], color="crimson", linewidth=2)
point_h, = ax.plot([], [], [], "o", color="navy")
point_q, = ax.plot([], [], [], "o", color="darkorange")
ax.legend()
set_axes_equal_3d(ax)


def animate(i):
    k = int(i / 100 * (len(ts) - 1))
    segment.set_data([H[0, k], Q[0, k]], [H[1, k], Q[1, k]])
    segment.set_3d_properties([H[2, k], Q[2, k]])
    point_h.set_data([H[0, k]], [H[1, k]])
    point_h.set_3d_properties([H[2, k]])
    point_q.set_data([Q[0, k]], [Q[1, k]])
    point_q.set_3d_properties([Q[2, k]])
    ax.view_init(elev=24, azim=35 + i)
    return segment, point_h, point_q


anim = FuncAnimation(fig, animate, frames=101, interval=70)
finish_animation(anim, fig)